In [12]:
from torchvision.datasets import ImageFolder
from torchvision.transforms import transforms
from torch.utils.data import DataLoader
import torch
import wandb

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.45, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = ImageFolder('../data/CitrusUAT_split_images/train', transform=transform)
val_dataset = ImageFolder('../data/CitrusUAT_split_images/val', transform=transform)
test_dataset = ImageFolder('../data/CitrusUAT_split_images/test', transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

def train(model, train_loader, val_loader, criterion, optimizer, num_epochs):
    # Determine whether to use GPU (if available) or CPU
    device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")


    for epoch in range(num_epochs):
        # Set the model to training mode
        model.train()

        # Initialize running loss and correct predictions count for training
        running_loss = 0.0
        running_corrects = 0

        # Iterate over the training data loader
        for inputs, labels in train_loader:
            # Move inputs and labels to the device (GPU or CPU)
            inputs = inputs.to(device)
            labels = labels.to(device)

            # Reset the gradients to zero before the backward pass
            optimizer.zero_grad()

            # Forward pass: compute the model output
            outputs = model(inputs)
            # Get the predicted class (with the highest score)
            _, preds = torch.max(outputs, 1)
            # Compute the loss between the predictions and actual labels
            loss = criterion(outputs, labels)

            # Backward pass: compute gradients
            loss.backward()
            # Perform the optimization step to update model parameters
            optimizer.step()

            # Accumulate the running loss and the number of correct predictions
            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

        # Compute average training loss and accuracy for this epoch
        train_loss = running_loss / len(train_loader.dataset)
        train_acc = running_corrects.float() / len(train_loader.dataset)

        # Set the model to evaluation mode for validation
        model.eval()
        # Initialize running loss and correct predictions count for validation
        running_loss = 0.0
        running_corrects = 0

        # Disable gradient computation for validation (saves memory and computations)
        with torch.no_grad():
            # Iterate over the validation data loader
            for inputs, labels in val_loader:
                # Move inputs and labels to the device (GPU or CPU)
                inputs = inputs.to(device)
                labels = labels.to(device)

                # Forward pass: compute the model output
                outputs = model(inputs)
                # Get the predicted class (with the highest score)
                _, preds = torch.max(outputs, 1)
                # Compute the loss between the predictions and actual labels
                loss = criterion(outputs, labels)

                # Accumulate the running loss and the number of correct predictions
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

        # Compute average validation loss and accuracy for this epoch
        val_loss = running_loss / len(val_loader.dataset)
        val_acc = running_corrects.float() / len(val_loader.dataset)

        # 2. Log Metrics to WandB
        wandb.log({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc
        })

        print(f'Epoch [{epoch+1}/{num_epochs}], train loss: {train_loss:.4f}, train acc: {train_acc:.4f}, val loss: {val_loss:.4f}, val acc: {val_acc:.4f}')


from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def evaluate_model(model, test_loader, device):
    # Initialize dictionaries to store correct and total predictions
    correct_pred = {classname: 0 for classname in test_loader.dataset.classes}
    total_pred = {classname: 0 for classname in test_loader.dataset.classes}

    # Set the model to evaluation mode
    model.eval()

    # Track the ground truth labels and predictions
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for inputs, labels in test_loader:
            # Move the inputs and labels to the device
            inputs = inputs.to(device)
            labels = labels.to(device)

            # Forward pass
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)

            # Collect predictions and labels for metric calculations
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

            # Update the correct and total predictions
            for label, prediction in zip(labels, preds):
                classname = test_loader.dataset.classes[label]
                if label == prediction:
                    correct_pred[classname] += 1
                total_pred[classname] += 1

    # Calculate accuracy per class
    accuracy_per_class = {classname: correct_pred[classname] / total_pred[classname] if total_pred[classname] > 0 else 0
                          for classname in test_loader.dataset.classes}

    # Calculate overall accuracy
    overall_accuracy = accuracy_score(all_labels, all_preds)


    # Print the evaluation results
    print("Accuracy per class:")
    for classname, accuracy in accuracy_per_class.items():
        print(f"{classname}: {accuracy:.4f}")

    print()
    print(f"Overall Accuracy: {overall_accuracy:.4f}")

In [ ]:
import numpy as np
import pandas as pd
import torch
import torchvision.models as models
from transformers import CLIPProcessor, CLIPModel, AutoModel, AutoImageProcessor
from tqdm import tqdm
import wandb

# RESNET -----------------------
resnet = models.resnet50(pretrained=True)
num_classes = 12
resnet.fc = torch.nn.Linear(resnet.fc.in_features, num_classes)

# Define loss function and optimizer
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(resnet.fc.parameters(), lr=0.001, momentum=0.9)

# 1. Login and Initialize
wandb.login()

run = wandb.init(
    entity="rchan192-university-of-california-riverside",
    project="icl_hlbdetection",
    config={
        "learning_rate": 0.001,
        "architecture": "resnet",
        "dataset": "CitrusUAT",
        "epochs": 1,
        "batch_size": 32
    }
)

# Run training
model = resnet.to(device)
train(model, train_loader, val_loader, criterion, optimizer, num_epochs=1)

# 3. Close the WandB run
run.finish()

evaluate_model(model, test_loader, device)

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


Epoch [1/1], train loss: 2.2757, train acc: 0.2402, val loss: 1.9584, val acc: 0.6013


epoch,▁
train_acc,▁
train_loss,▁
val_acc,▁
val_loss,▁
epoch,1
train_acc,0.24016
train_loss,2.27571
val_acc,0.60131
val_loss,1.95836


Accuracy per class:
Citrus_leafminer: 0.4000
Fe: 0.7778
Greasy_spot: 1.0000
HLB: 0.0000
Healthy: 0.7647
Mg: 0.8500
Mn: 0.0000
N: 0.0000
Red_scale: 0.0000
Red_scale_sequelae: 0.3043
Texas_mite: 1.0000
Zn: 0.9500

Overall Accuracy: 0.6178


In [16]:
import numpy as np
import pandas as pd
import torch
import torchvision.models as models
from transformers import CLIPProcessor, CLIPModel, AutoModel, AutoImageProcessor
from tqdm import tqdm
import wandb

# VGG19 -----------------------
# Model setup
vgg19 = models.vgg19(pretrained=True)
num_classes = len(train_dataset.classes)
vgg19.classifier[6] = torch.nn.Linear(4096, num_classes)
model = vgg19.to(device)

# Define loss function and optimizer
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

# 1. Login and Initialize
wandb.login()

run = wandb.init(
    entity="rchan192-university-of-california-riverside",
    project="icl_hlbdetection",
    config={
        "learning_rate": 0.001,
        "architecture": "vgg19",
        "dataset": "CitrusUAT",
        "epochs": 15,
        "batch_size": 32
    }
)

# Run training
model = vgg19.to(device)
train(model, train_loader, val_loader, criterion, optimizer, num_epochs=15)

# 3. Close the WandB run
run.finish()

evaluate_model(model, test_loader, device)

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


epoch,▁▃▆█
train_acc,▁▆▇█
train_loss,█▃▂▁
val_acc,▁▄▅█
val_loss,█▄▃▁
epoch,4
train_acc,0.94094
train_loss,0.16078
val_acc,0.98039
val_loss,0.08384


Epoch [1/15], train loss: 1.8379, train acc: 0.4016, val loss: 0.6858, val acc: 0.7843
Epoch [2/15], train loss: 0.5257, train acc: 0.8123, val loss: 0.2615, val acc: 0.8954
Epoch [3/15], train loss: 0.2583, train acc: 0.9213, val loss: 0.2539, val acc: 0.9150
Epoch [4/15], train loss: 0.1918, train acc: 0.9396, val loss: 0.1245, val acc: 0.9608
Epoch [5/15], train loss: 0.0886, train acc: 0.9724, val loss: 0.0452, val acc: 0.9804
Epoch [6/15], train loss: 0.1348, train acc: 0.9724, val loss: 0.0657, val acc: 0.9869
Epoch [7/15], train loss: 0.0576, train acc: 0.9816, val loss: 0.0268, val acc: 0.9869
Epoch [8/15], train loss: 0.0153, train acc: 0.9948, val loss: 0.0176, val acc: 0.9935
Epoch [9/15], train loss: 0.0059, train acc: 1.0000, val loss: 0.0015, val acc: 1.0000
Epoch [10/15], train loss: 0.0022, train acc: 0.9987, val loss: 0.0002, val acc: 1.0000
Epoch [11/15], train loss: 0.0007, train acc: 1.0000, val loss: 0.0002, val acc: 1.0000
Epoch [12/15], train loss: 0.0010, train 

epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
train_acc,▁▆▇▇███████████
train_loss,█▃▂▂▁▂▁▁▁▁▁▁▁▁▁
val_acc,▁▅▅▇▇█████████▇
val_loss,█▄▄▂▁▂▁▁▁▁▁▁▁▁▁
epoch,15
train_acc,0.97638
train_loss,0.07626
val_acc,0.97386
val_loss,0.04719


Accuracy per class:
Citrus_leafminer: 1.0000
Fe: 0.9630
Greasy_spot: 1.0000
HLB: 0.9167
Healthy: 1.0000
Mg: 1.0000
Mn: 0.4286
N: 1.0000
Red_scale: 1.0000
Red_scale_sequelae: 0.9565
Texas_mite: 0.7000
Zn: 1.0000

Overall Accuracy: 0.9319
